In [ ]:
%cd D:\Uni\Master\Data Science\Project\Code_local\MOFA
#%pip install lifelines

In [ ]:
print("This generates the file data/HumanMethylation450_filtered.csv in the MOFA/data directory. dont run if you already have it.")
import pandas as pd
import os


# This file contains the methylation probes we excluded due to sex effect.
filter_list_csv = "methylation_probes_to_drop.csv" 

#raw_meth_file = "../HumanMethylation450.txt" 
raw_meth_file = os.path.join(os.getcwd(), "..", "TCGA-KIRC/HumanMethylation450.txt")

os.makedirs("data", exist_ok=True)

cleaned_meth_output = "data/HumanMethylation450_filtered.csv"

print("Loading filtering blacklist...")
filter_df = pd.read_csv(filter_list_csv)

probes_to_remove = set(filter_df['probe'].astype(str).tolist())
print(f"Loaded {len(probes_to_remove)} distinct probes to filter out.")

print("\nLoading raw methylation matrix (takes a few minutes)...")

meth_df = pd.read_csv(raw_meth_file, sep="\t", index_col=0)

print(f"Original matrix shape: {meth_df.shape[0]} features (probes) x {meth_df.shape[1]} samples.")

existing_probes = meth_df.index 

matching_probes = probes_to_remove.intersection(set(existing_probes))
print(f"Found {len(matching_probes)} blacklisted probes matching your dataset matrix.")

if len(matching_probes) > 0:
    cleaned_meth_df = meth_df[~meth_df.index.astype(str).isin(probes_to_remove)]    
    print(f"Filtering complete")
    print(f"New matrix shape:      {cleaned_meth_df.shape[0]} features x {cleaned_meth_df.shape[1]} samples.")
    print(f"Successfully dropped: {meth_df.shape[0] - cleaned_meth_df.shape[0]} rows.")
else:
    print("Warning: No matching probes from your CSV file were found in your methylation matrix index. Please write us in this case, you may have the directory set up wrongly.")
    cleaned_meth_df = meth_df

df_transposed = cleaned_meth_df.T

df_transposed.index.name = "Sample_ID"

print(f"\nSaving clean matrix to: {cleaned_meth_output} (This takes even longer, dont worry, it didnt crash. Took me ~ 10 minutes)")
df_transposed.to_csv(cleaned_meth_output, sep="\t")
print("Saving is finished. This file is used instead of the full methylation matrix due to the sex relevant genes, that scew the entire model.")

In [ ]:
print("This trains the MOFA model using 3 layers and filtered methylation data. This takes ~3 hours, dont run if you already copied the mofa_kidney_filtered_model.hdf5 file. Size is ~1.2 GB")
import os
import numpy as np
import pandas as pd
from mofapy2.run.entry_point import entry_point


file_paths = {
    "RNAseq": os.path.join(os.getcwd(), "..", "TCGA-KIRC/HiSeqV2.txt"),
    #"Exon": "D:/Uni/Master/Data Science/Project/TCGA_Kidney/HiSeqV2_exon",
    #"Methylation": "D:/Uni/Master/Data Science/Project/TCGA_Kidney/HumanMethylation450",
    "Methylation": "data/HumanMethylation450_filtered.csv",
    "RPPA": os.path.join(os.getcwd(), "..", "TCGA-KIRC/RPPA")
    #"CNV": "D:/Uni/Master/Data Science/Project/TCGA_Kidney/Gistic2_CopyNumber_Gistic2_all_thresholded_by_genes"
}

loaded_dfs = {}

def process_tcga_file(filepath):
    df = pd.read_csv(filepath, sep="\t", index_col=0)
    columns_are_samples = df.columns.astype(str).str.contains("TCGA-").any()
    rows_are_samples = df.index.astype(str).str.contains("TCGA-").any()
    
    if columns_are_samples:
        df = df.T
    elif not rows_are_samples:
        raise ValueError(f"Could not automatically locate TCGA sample identifiers in {filepath}, please write us in case the directories are set up wrongly")
        
    df.index.name = "Sample_ID"
    
    # Clean up duplicate sample rows by averaging numeric values. This is just in case, but there are no duplicates in our current values.
    df = df.groupby(level=0).mean()
    return df

for view_name, path in file_paths.items():
    if os.path.exists(path):
        print(f"Processing layer: {view_name} from {path}...")
        loaded_dfs[view_name] = process_tcga_file(path)
    else:
        print(f"Warning: File {path} not found. Please verify local directory paths.")


#GROUPS AND LAYERS GENERATION FOR MULTI-GROUP MOFA

all_samples = set()
for df in loaded_dfs.values():
    all_samples.update(df.index.tolist())
all_samples = list(all_samples)

# Map samples into structured condition group categories:
# -01 to -09 -> Disease tissue, -11 to -19 -> Adjacent Normal tissue. there is one 05 sample with additional new cancer, but we just handle it as diseased here.
sample_groups = {}
for sample in all_samples:
    suffix = sample.split("-")[-1]
    if suffix.startswith("0"):
        sample_groups[sample] = "Disease"
    elif suffix.startswith("1"):
        sample_groups[sample] = "Normal"
    else:
        sample_groups[sample] = "Other"

# Retain only clear Disease and Normal samples
valid_samples = [s for s in all_samples if sample_groups[s] in ["Disease", "Normal"]]

print("We define the views and groups for the mofa training input.")
views_names = list(loaded_dfs.keys())
groups_names = ["Disease", "Normal"]

print(f"\nConfigured Views: {views_names}")
print(f"Configured Groups: {groups_names}")

data = [[None for _ in range(len(groups_names))] for _ in range(len(views_names))]
samples_names = [[] for _ in range(len(groups_names))]

# We create unique suffix tag mapping to prevent feature space overlaps. We haven't seen any overlapping naming, but just in case.
feature_suffixes = {
    "RNAseq": "_rna", 
    #"Exon": "_exon", 
    "Methylation": "_meth", 
    "RPPA": "_rppa", 
    #"CNV": "_cnv"
}

features_names = []
for view_name in views_names:
    suffix = feature_suffixes.get(view_name, f"_{view_name.lower()}")
    v_features = [f"{feat}{suffix}" for feat in loaded_dfs[view_name].columns]
    features_names.append(v_features)

for g_idx, group_name in enumerate(groups_names):
    group_samples = [s for s in valid_samples if sample_groups[s] == group_name]
    samples_names[g_idx] = group_samples
    
    for v_idx, view_name in enumerate(views_names):
        aligned_df = loaded_dfs[view_name].reindex(group_samples)
        data[v_idx][g_idx] = aligned_df.to_numpy()

print("All 3 multi-omics data layers are structured and ready for MOFA+ integration!")

# MOFA2 INITIALIZATION & TRAINING

ent = entry_point()

ent.set_data_options(scale_views=False)

ent.set_data_matrix(
    data,
    views_names=views_names,
    groups_names=groups_names,
    samples_names=samples_names,
    features_names=features_names,
    likelihoods=["gaussian"] * len(views_names),
)

print("We set model training priors here with ARD(Automatic Relevance Determination) weights enabled, meaning it can go lower from our set 15 factors, if the explained variance is too low.")
ent.set_model_options(factors=15, spikeslab_weights=True, ard_weights=True)

#Here we set the setting for training the model itself, with how fast its supposed to go. Since it takes a lot of memory, GPU is turned to false. Also, its REALLY annoying to set everything up with cupy to work with GPU.
ent.set_train_options(
    iter=1000, convergence_mode="fast", dropR2=None, gpu_mode=FALSE, seed=42
)

print("MOFA is building...")
ent.build()
print("MOFA is running...")
ent.run()
print("MOFA finished running!")

output_model_name = "mofa_kidney_filtered_model.hdf5"
ent.save(output_model_name)
print(f"\nModel training finished successfully! Saved as: {output_model_name}")